In [2]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import time
import warnings

import joblib
import numpy as np
import polars as pl
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

warnings.filterwarnings('ignore', message='X does not have valid feature names')
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

TARGET_COL = 'label'
N_REPEATS = 3
WARMUP_SIZE = 1000


I0000 00:00:1779298330.081186   10995 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779298330.081777   10995 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779298330.134054   10995 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779298331.480068   10995 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [4]:
# Carga de dataset y preprocesado, igual que en los cuadernos de ataques para IOT-23
prepared_path = '../../DATASETS/dataSets_Reducidos/iot-23/datos_IOT_23_preparado.csv'

df_encoded = pl.read_csv(prepared_path)

feature_columns = [col for col in df_encoded.columns if col != TARGET_COL]
X = df_encoded.select(feature_columns)
y_np = df_encoded[TARGET_COL].to_numpy().astype(np.int8)
X_np = X.to_numpy().astype(np.float32)

indices = np.arange(X_np.shape[0])

train_full_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=SEED,
    stratify=y_np,
)

X_full_train_np = X_np[train_full_idx]
X_test_np = X_np[test_idx]

X_full_base = X_np.astype(np.float32)

mlp_scaler = StandardScaler()
mlp_scaler.fit(X_full_train_np)
X_full_scaled_mlp = mlp_scaler.transform(X_full_base).astype(np.float32)

cnn_scaler = MinMaxScaler()
cnn_scaler.fit(X_full_train_np)
X_full_scaled_cnn = cnn_scaler.transform(X_full_base).astype(np.float32)
X_full_cnn = X_full_scaled_cnn.reshape(X_full_scaled_cnn.shape[0], X_full_scaled_cnn.shape[1], 1)

print(f'Train full: {len(X_full_train_np):,} muestras')
print(f'Test:       {len(X_test_np):,} muestras')
print(f'Total:      {len(X_full_base):,} muestras')
print(f'Features:   {X_full_base.shape[1]}')


Train full: 931,880 muestras
Test:       232,971 muestras
Total:      1,164,851 muestras
Features:   22


In [6]:
# RF

rf_path = '../model/iot-23/rf_iot23.joblib'

rf_model = joblib.load(rf_path)
print('Modelo cargado')

# Warm-up
_ = rf_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_rf = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = rf_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_rf.append(t1 - t0)

tiempo_total_rf = float(np.mean(tiempos_rf))

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_rf]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_rf:.4f} s')


Modelo cargado
Tiempos medidos: [0.8168, 0.9302, 0.9857]
Tiempo medio de inferencia sobre todo el dataset: 0.9109 s


In [ ]:
# XGBOOST

xgb_path = '../model/iot-23/xgb_iot23.joblib'

xgb_model = joblib.load(xgb_path)
print('Modelo cargado')

# Warm-up
_ = xgb_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_xgb = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = xgb_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_xgb.append(t1 - t0)

tiempo_total_xgb = float(np.mean(tiempos_xgb))

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_xgb]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_xgb:.4f} s')


In [ ]:
# LIGHTGBM

lgbm_path = '../model/iot-23/lgbm_iot23.joblib'

lgbm_model = joblib.load(lgbm_path)
print('Modelo cargado')

# Warm-up
_ = lgbm_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_lgbm = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = lgbm_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_lgbm.append(t1 - t0)

tiempo_total_lgbm = float(np.mean(tiempos_lgbm))

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_lgbm]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_lgbm:.4f} s')


In [ ]:
# CATBOOST

catboost_path = '../model/iot-23/catboost_iot23.joblib'

catboost_model = joblib.load(catboost_path)
print('Modelo cargado')

# Warm-up
_ = catboost_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_catboost = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = catboost_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_catboost.append(t1 - t0)

tiempo_total_catboost = float(np.mean(tiempos_catboost))

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_catboost]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_catboost:.4f} s')


In [ ]:
# SVM

svm_path = '../model/iot-23/svm_iot23.joblib'

svm_model = joblib.load(svm_path)
print('Modelo cargado')

# Warm-up
_ = svm_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_svm = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = svm_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_svm.append(t1 - t0)

tiempo_total_svm = float(np.mean(tiempos_svm))

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_svm]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_svm:.4f} s')


In [ ]:
# MLP

mlp_path = '../model/iot-23/mlp_iot23.joblib'

mlp_model = joblib.load(mlp_path)
print('Modelo cargado')

# Warm-up
_ = mlp_model.predict(X_full_scaled_mlp[:min(WARMUP_SIZE, len(X_full_scaled_mlp))], batch_size=4096, verbose=0)

# Medida latencia
tiempos_mlp = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = mlp_model.predict(X_full_scaled_mlp, batch_size=4096, verbose=0)
    t1 = time.perf_counter()
    tiempos_mlp.append(t1 - t0)

tiempo_total_mlp = float(np.mean(tiempos_mlp))

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_mlp]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_mlp:.4f} s')


In [ ]:
# CNN

cnn_path = '../model/iot-23/cnn_iot23.joblib'

cnn_model = joblib.load(cnn_path)
print('Modelo cargado')

# Warm-up
_ = cnn_model.predict(X_full_cnn[:min(WARMUP_SIZE, len(X_full_cnn))], batch_size=4096, verbose=0)

# Medida latencia
tiempos_cnn = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = cnn_model.predict(X_full_cnn, batch_size=4096, verbose=0)
    t1 = time.perf_counter()
    tiempos_cnn.append(t1 - t0)

tiempo_total_cnn = float(np.mean(tiempos_cnn))

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_cnn]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_cnn:.4f} s')
